In [1]:
import numpy as np
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from services.files import get_files
from services.distortion import hard_clip
from services.distortion import soft_clip
from services.distortion import bitcrush
from services.distortion import wavefold
from services.distortion import tremolo
from services.distortion import reverb
from sklearn.model_selection import train_test_split
import os
import librosa
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.preprocessing import StandardScaler
import time
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
  
import seaborn as sns
import matplotlib.pyplot as plt

In [27]:

torch.set_default_device('cpu')


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [28]:
files = get_files()
print(len(files))
print(files[:5])

2913
['TinySOL\\Brass\\Bass_Tuba\\ordinario\\BTb-ord-A#1-ff-N-T30d.wav', 'TinySOL\\Brass\\Bass_Tuba\\ordinario\\BTb-ord-A#1-mf-N-N.wav', 'TinySOL\\Brass\\Bass_Tuba\\ordinario\\BTb-ord-A#1-pp-N-N.wav', 'TinySOL\\Brass\\Bass_Tuba\\ordinario\\BTb-ord-A#2-ff-N-T19u.wav', 'TinySOL\\Brass\\Bass_Tuba\\ordinario\\BTb-ord-A#2-mf-N-T29u.wav']


In [29]:
AudioData = []
TargetSeconds = 3
TargetSamples = TargetSeconds * 44100 

for file in files:
    audio, sr = librosa.load(file, sr=44100)
    audio = audio/ np.max(np.abs(audio))
    audio_fixed = librosa.util.fix_length(audio, size=TargetSamples)
    AudioData.append(audio_fixed)


AudioData = np.array(AudioData)

print(AudioData.shape)

print(AudioData,AudioData[0].shape, AudioData[1].shape)

(2913, 132300)
[[ 0.0000000e+00  0.0000000e+00  1.4452955e-04 ...  0.0000000e+00
   0.0000000e+00  0.0000000e+00]
 [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 ... -2.1752988e-01
  -2.2788845e-01 -2.3585658e-01]
 [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 ... -1.0321101e-01
  -1.0550459e-01 -1.0550459e-01]
 ...
 [ 0.0000000e+00 -1.6079756e-04 -1.6079756e-04 ... -7.2809136e-01
  -7.1410197e-01 -5.9093100e-01]
 [ 0.0000000e+00  0.0000000e+00  2.3849272e-04 ... -3.5606965e-01
  -2.5017887e-01 -7.0593849e-02]
 [ 1.1376564e-03  0.0000000e+00  0.0000000e+00 ... -5.9726965e-01
  -5.6882823e-01 -5.3924912e-01]] (132300,) (132300,)


In [30]:
X_train, X_test = train_test_split(
    AudioData, test_size=0.2, random_state=42, shuffle=True
)

In [31]:
X_train = X_train.reshape(len(X_train), -1)
X_test  = X_test.reshape(len(X_test), -1)

In [32]:
def apply_random_distortion(x):
    effects = [
        lambda x: hard_clip(x, 0.5),
        lambda x: soft_clip(x, 3),
        lambda x: bitcrush(x),
        lambda x: wavefold(x),
        lambda x: tremolo(x),
        lambda x: reverb(x)
    ]
    
    effect = np.random.choice(effects)
    distorted = effect(x)

    
    distorted = distorted / (np.max(np.abs(distorted)) + 1e-8)

    return distorted

In [33]:
def build_binary_dataset(clean_data):
    X = []
    y = []

    for signal in clean_data:
        #Clean sample
        X.append(signal)
        y.append(0)

        #Distorted sample
        distorted = apply_random_distortion(signal)
        X.append(distorted)
        y.append(1)

    X = np.array(X)
    y = np.array(y)

    #Shuffle
    perm = np.random.permutation(len(X))
    return X[perm], y[perm]

In [34]:
X_train, y_train = build_binary_dataset(X_train_clean)
X_test, y_test = build_binary_dataset(X_test_clean)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(4660, 132300) (4660,)
(1166, 132300) (1166,)


In [35]:
def extract_features(X):
    features = []

    for signal in X:
        
        mean = np.mean(signal)
        std = np.std(signal)
        energy = np.mean(np.abs(signal))
        rms = np.sqrt(np.mean(signal**2))
        peak = np.max(np.abs(signal))

        
        crest = peak / (rms + 1e-8)
        zcr = np.mean(np.abs(np.diff(np.sign(signal))))

        
        fft = np.fft.rfft(signal)
        mag = np.abs(fft)

        fft_mean = np.mean(mag)
        fft_std = np.std(mag)

        
        freqs = np.fft.rfftfreq(len(signal), d=1/44100)
        centroid = np.sum(freqs * mag) / (np.sum(mag) + 1e-8)
        bandwidth = np.sqrt(np.sum(((freqs - centroid)**2) * mag) / (np.sum(mag) + 1e-8))

        features.append([
            mean, std, energy, rms, peak,
            crest, zcr,
            fft_mean, fft_std,
            centroid, bandwidth
        ])

    return np.array(features)

In [36]:
X_train_feat = extract_features(X_train)
X_test_feat  = extract_features(X_test)

In [37]:
scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_test_feat  = scaler.transform(X_test_feat)

In [38]:
X_train = X_train.reshape(len(X_train), -1)
X_test  = X_test.reshape(len(X_test), -1)



In [39]:
model = SGDClassifier(loss="log_loss", max_iter=1, warm_start=True)
epochs = 100

for epoch in range(epochs):
    model.fit(X_train_feat, y_train)
    
    train_acc = model.score(X_train_feat, y_train)
    test_acc = model.score(X_test_feat, y_test)
    
    print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

c:\Dev\Projects\Distortion_ML\venv\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Dev\Projects\Distortion_ML\venv\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Dev\Projects\Distortion_ML\venv\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Dev\Projects\Distortion_ML\venv\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Dev\Projects\Dist

Epoch 1/100 | Train Acc: 0.6693 | Test Acc: 0.6647
Epoch 2/100 | Train Acc: 0.7148 | Test Acc: 0.7341
Epoch 3/100 | Train Acc: 0.6910 | Test Acc: 0.7101
Epoch 4/100 | Train Acc: 0.7219 | Test Acc: 0.7213
Epoch 5/100 | Train Acc: 0.7288 | Test Acc: 0.7376
Epoch 6/100 | Train Acc: 0.6526 | Test Acc: 0.6732
Epoch 7/100 | Train Acc: 0.6974 | Test Acc: 0.7024
Epoch 8/100 | Train Acc: 0.7146 | Test Acc: 0.7316
Epoch 9/100 | Train Acc: 0.7264 | Test Acc: 0.7504
Epoch 10/100 | Train Acc: 0.7116 | Test Acc: 0.7384
Epoch 11/100 | Train Acc: 0.6796 | Test Acc: 0.6930
Epoch 12/100 | Train Acc: 0.6914 | Test Acc: 0.7144
Epoch 13/100 | Train Acc: 0.6979 | Test Acc: 0.7127
Epoch 14/100 | Train Acc: 0.6923 | Test Acc: 0.7084
Epoch 15/100 | Train Acc: 0.6779 | Test Acc: 0.6861
Epoch 16/100 | Train Acc: 0.6749 | Test Acc: 0.6664
Epoch 17/100 | Train Acc: 0.6923 | Test Acc: 0.6938
Epoch 18/100 | Train Acc: 0.7157 | Test Acc: 0.7213
Epoch 19/100 | Train Acc: 0.7290 | Test Acc: 0.7358
Epoch 20/100 | Train 

c:\Dev\Projects\Distortion_ML\venv\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Dev\Projects\Distortion_ML\venv\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Dev\Projects\Distortion_ML\venv\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Dev\Projects\Distortion_ML\venv\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Dev\Projects\Dist

In [40]:
y_pred = model.predict(X_test_feat)

print("Accuracy:", accuracy_score(y_test, y_pred) * 100 , "%")
print("\nReport:\n", classification_report(y_test, y_pred))

Accuracy: 73.58490566037736 %

Report:
               precision    recall  f1-score   support

           0       0.75      0.71      0.73       583
           1       0.72      0.76      0.74       583

    accuracy                           0.74      1166
   macro avg       0.74      0.74      0.74      1166
weighted avg       0.74      0.74      0.74      1166



In [41]:
torch.save(model, "logistic_model.pth")